In [ ]:
# 13_pooled_vs_separate_model.ipynb
# Appendix E3: sex-stratified vs pooled (sex-as-feature) model on the identical
# train/test partition. Reproduces the published per-sex AUCs as a validation check,
# then fits a pooled XGBoost and reports AUC / PR-AUC / Brier overall and per sex.
# Runs locally/cloud; no API key. Writes results/tables/table_e3_pooled_vs_separate.csv.
import json
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA, MODELS, TAB = ROOT/"data", ROOT/"models", ROOT/"results"/"tables"
SEED = 42
FEATS = ["age","HE_BMI","HE_wc","HE_wt","HE_ht","N_EN","N_CHO","N_SUGAR","N_CHOL",
         "sm_presnt","dr_month","pa_aerobic","L_BR_FQ","edu","ho_incm","D_1_1"]
par = {s: json.load(open(MODELS/f"xgb_{s}_params.json")) for s in ("male","female")}
split = {s: pd.read_parquet(DATA/f"split_{s}.parquet") for s in ("male","female")}
def fit_eval(train, test, params, feats):
    clf = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4)
    clf.fit(train[feats], train["diabetes"])
    p = clf.predict_proba(test[feats])[:,1]; y = test["diabetes"].values
    return p, dict(AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p), Brier=brier_score_loss(y,p))
# validation: stratified
pm,mm = fit_eval(split["male"][split["male"].split=="train"], split["male"][split["male"].split=="test"], par["male"], FEATS)
pf,mf = fit_eval(split["female"][split["female"].split=="train"], split["female"][split["female"].split=="test"], par["female"], FEATS)
print("Stratified male  :", {k:round(v,3) for k,v in mm.items()})
print("Stratified female:", {k:round(v,3) for k,v in mf.items()})
# pooled
for s in ("male","female"):
    split[s]["sex_male"] = (split[s]["sex"].astype(str).str.lower().str.startswith("m")).astype(int)
FP = FEATS + ["sex_male"]
train = pd.concat([split["male"][split["male"].split=="train"], split["female"][split["female"].split=="train"]], ignore_index=True)
test  = pd.concat([split["male"][split["male"].split=="test"],  split["female"][split["female"].split=="test"]],  ignore_index=True)
pp, mp = fit_eval(train, test, par["male"], FP)   # male-tuned hyperparameters used for the pooled fit
rows = [["Sex-stratified","Male",*[mm[k] for k in ("AUC","PR_AUC","Brier")]],
        ["Sex-stratified","Female",*[mf[k] for k in ("AUC","PR_AUC","Brier")]]]
yall = np.concatenate([split["male"][split["male"].split=="test"]["diabetes"].values,
                       split["female"][split["female"].split=="test"]["diabetes"].values])
pall = np.concatenate([pm, pf])
rows.append(["Sex-stratified","Combined", roc_auc_score(yall,pall), average_precision_score(yall,pall), brier_score_loss(yall,pall)])
rows.append(["Pooled (sex feature)","Overall", mp["AUC"], mp["PR_AUC"], mp["Brier"]])
for v,lab in [(1,"Male subset"),(0,"Female subset")]:
    m = test["sex_male"].values==v; y=test["diabetes"].values[m]; ps=pp[m]
    rows.append(["Pooled (sex feature)", lab, roc_auc_score(y,ps), average_precision_score(y,ps), brier_score_loss(y,ps)])
out = pd.DataFrame(rows, columns=["Specification","Scope","AUC","PR_AUC","Brier"]).round(3)
out.to_csv(TAB/"table_e3_pooled_vs_separate.csv", index=False); print(out.to_string(index=False))
